### Installations and Imports

In [ ]:
import pandas as pd
import re
import random
import io
import time
from openai import OpenAI
from google.colab import files
from IPython.display import display

### API Config

In [ ]:
# Abby's comment: should we use claude API to help with creating the less/highly quantified versions of the resume?

### NOTE: Anthropic offers $5 free credits when you create an API account, which should be more than enough for 60 rewrite calls on 30 resumes.

### CODE:
# import anthropic
# client = anthropic.Anthropic(api_key="YOUR_ANTHROPIC_API_KEY_HERE")
# MODEL  = "claude-haiku-4-5-20251001"
# def call_llm(prompt: str) -> str:
#     response = client.messages.create(
#         model=MODEL, max_tokens=2048,
#         messages=[{"role": "user", "content": prompt}]
#     )
#     return response.content[0].text.strip()

### ADDITIONAL NOTE: use Google Colab's secret manager instead of pasting API key directly
# from google.colab import userdata
# client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

### Load CSV files

In [ ]:
import pandas as pd
import re
import random
import io
from google.colab import files
from IPython.display import display

print('Please upload your two CSV files:')
uploaded = files.upload()

filenames    = list(uploaded.keys())
resumes_file = next(f for f in filenames if 'base' in f or 'resume' in f.lower())
unis_file    = next(f for f in filenames if 'unis' in f.lower() or 'dataset' in f.lower())

df_resumes = pd.read_csv(io.BytesIO(uploaded[resumes_file]))
df_lists   = pd.read_csv(io.BytesIO(uploaded[unis_file]))

print(f'✓ Resumes loaded : {len(df_resumes)} rows')
print(f'✓ Lists loaded   : {len(df_lists)} rows')

Please upload your two CSV files:


Saving unis_companies_dataset.csv to unis_companies_dataset.csv
Saving base_resumes.csv to base_resumes.csv
✓ Resumes loaded : 30 rows
✓ Lists loaded   : 40 rows


### Preview Raw Data

In [ ]:
print('=== Base Resumes ===')
display(df_resumes.head(3))
display(df_resumes.tail(3))

print('\n=== Universities & Companies List ===')
display(df_lists.head(3))
display(df_lists.tail(3))

=== Base Resumes ===


,Category,Quantification,Gender,Ethnicity,Resume
0,Software Engineer,Highly quantified,Female,American Indian,Name: Aiyana Morningstar; Skills - Front-end S...
1,Software Engineer,Low quantified,Female,Asian,Name: Li Mei Chen; Experience 1 - Company: Com...
2,Software Engineer,Low quantified,Female,Black,Name: Aaliyah Thompson; Skills - Proficient: A...


,Category,Quantification,Gender,Ethnicity,Resume
27,Product Manager,Highly quantified,Male,Black,"Name: Kofi Asante; Summary: Customer-focused, ..."
28,Product Manager,Highly quantified,Male,Pacific Islander,Name: Tane Faleolo; Skills - Product: Roadmapp...
29,Product Manager,Highly quantified,Male,White,Name: Liam O'Brien; Summary: Product Professio...



=== Universities & Companies List ===


,Name,Type,Credentials
0,Carnegie Mellon University,University,Elite
1,Univ. of Illinois at Urbana-Champaign,University,Elite
2,Univ. of California - San Diego,University,Elite


,Name,Type,Credentials
37,ScaleUp,Company,Non-elite
38,LaunchPad,Company,Non-elite
39,CodeNest,Company,Non-elite


## PHASE 1: GENERATE CREDENTIAL VARIANTS

### Build Credential Lookup List

In [ ]:
elite_unis    = df_lists[(df_lists['Type'] == 'University') & (df_lists['Credentials'] == 'Elite')   ]['Name'].str.strip().tolist()
elite_cos     = df_lists[(df_lists['Type'] == 'Company')    & (df_lists['Credentials'] == 'Elite')   ]['Name'].str.strip().tolist()
nonelite_unis = df_lists[(df_lists['Type'] == 'University') & (df_lists['Credentials'] == 'Non-elite')]['Name'].str.strip().tolist()
nonelite_cos  = df_lists[(df_lists['Type'] == 'Company')    & (df_lists['Credentials'] == 'Non-elite')]['Name'].str.strip().tolist()

summary = pd.DataFrame({
    'Category'       : ['Universities', 'Companies'],
    'Elite'          : [elite_unis,     elite_cos],
    'Non-Elite'      : [nonelite_unis,  nonelite_cos],
})
display(summary)

,Category,Elite,Non-Elite
0,Universities,"[Carnegie Mellon University, Univ. of Illinois...","[University of Utah, Kansas State University, ..."
1,Companies,"[Meta, Netflix, Apple, Google, Amazon, Microso...","[StartupX, TechStartup, GrowthCo, InsightAI, B..."


### Define Replacement Functions

In [ ]:
def replace_institutions(text, unis_pool, seed=None):
    rng = random.Random(seed)
    pool = unis_pool.copy()
    rng.shuffle(pool)
    pool_cycle = pool * 10
    counter = [0]
    def replacer(match):
        replacement = pool_cycle[counter[0] % len(pool)]
        counter[0] += 1
        return f'Institution: {replacement}'
    return re.sub(r'Institution:\s*[^;]+?(?=;|$)', replacer, text)

def replace_companies(text, cos_pool, seed=None):
    rng = random.Random(seed)
    pool = cos_pool.copy()
    rng.shuffle(pool)
    pool_cycle = pool * 10
    counter = [0]
    def replacer(match):
        replacement = pool_cycle[counter[0] % len(pool)]
        counter[0] += 1
        return f'Company: {replacement}'
    return re.sub(r'Company:\s*[^;]+?(?=;|$)', replacer, text)

def make_resume_variant(text, unis_pool, cos_pool, seed=None):
    text = replace_institutions(text, unis_pool, seed=seed)
    text = replace_companies(text, cos_pool, seed=seed)
    return text

print('✓ Replacement functions defined')

✓ Replacement functions defined


### Apply Credential Replacements

In [ ]:
df_resumes['Resume_Elite'] = [
    make_resume_variant(row['Resume'], elite_unis, elite_cos, seed=idx)
    for idx, row in df_resumes.iterrows()
]

df_resumes['Resume_NonElite'] = [
    make_resume_variant(row['Resume'], nonelite_unis, nonelite_cos, seed=idx)
    for idx, row in df_resumes.iterrows()
]

print(f'✓ Generated Resume_Elite and Resume_NonElite for {len(df_resumes)} resumes')

✓ Generated Resume_Elite and Resume_NonElite for 30 resumes


### Credential Sanity Check

In [ ]:
sample_idx = 0  # ← change to check a different resume

def extract(text, field):
    pattern = rf'{field}:\s*([^;]+?)(?:;|$)'
    return re.findall(pattern, text)

rows = []
for field, label in [('Institution', 'University'), ('Company', 'Company')]:
    orig = extract(df_resumes.loc[sample_idx, 'Resume'],          field)
    eli  = extract(df_resumes.loc[sample_idx, 'Resume_Elite'],    field)
    ne   = extract(df_resumes.loc[sample_idx, 'Resume_NonElite'], field)
    for i, (o, e, n) in enumerate(zip(orig, eli, ne)):
        rows.append({'Field': f'{label} {i+1}', 'Original': o.strip(),
                     'Elite': e.strip(), 'Non-Elite': n.strip()})

sanity_df = pd.DataFrame(rows)
print(f'=== Sanity Check: Resume #{sample_idx} ===')
display(sanity_df)

=== Sanity Check: Resume #0 ===


,Field,Original,Elite,Non-Elite
0,University 1,WelTec Petone,Cornell University,Stockton University
1,University 2,Chapman University,University of Washington,University of Nebraska -- Lincoln
2,Company 1,Numeral Studio,NVIDIA,ScaleUp
3,Company 2,Habitual Genesis,Palantir,LaunchPad
4,Company 3,PartsTrader Markets Ltd,Netflix,TechStartup
5,Company 4,WorkTango (formerly KazooHR),Microsoft,OrbitX
6,Company 5,Conde Nast,Google,InsightAI
7,Company 6,Bypass Mobile,Amazon,Buildr
8,Company 7,Union Metrics,Apple,GrowthCo
9,Company 8,The Flatiron School,Meta,StartupX


### Save and Download

In [ ]:
output_filename = 'resumes_credential_variants.csv'
df_resumes[['Category', 'Quantification', 'Gender', 'Ethnicity',
            'Resume', 'Resume_Elite', 'Resume_NonElite']].to_csv(output_filename, index=False)

print(f'✓ Saved to {output_filename}')
files.download(output_filename)

✓ Saved to resumes_credential_variants.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## PHASE 2: GENERATE QUANTIFICATION VARIANTS
### NOTE: need to set up API LLM config first

### Define Rewrite Prompts    

In [ ]:
STRIP_PROMPT = """You are editing a resume. Your task is to remove quantified metrics from the responsibilities sections only.

Rules:
- Replace specific numbers, percentages, and counts with vague qualitative language.
  Examples: "increased sales by 40%" → "increased sales significantly"
            "reduced load time by 200ms" → "reduced load time noticeably"
            "managed a team of 8" → "managed a team"
- Do NOT change anything outside of Responsibilities fields (skills, titles, dates, technologies, education are untouched).
- Preserve the exact semicolon-separated format of the original text.
- Return only the rewritten resume text with no explanation or preamble.

Resume:
{resume}"""

ENHANCE_PROMPT = """You are editing a resume. Your task is to add plausible quantified metrics to the responsibilities sections only.

Rules:
- Add specific numbers, percentages, or counts to achievement statements where they are missing.
  Examples: "improved team efficiency" → "improved team efficiency by 30%"
            "reduced customer complaints" → "reduced customer complaints by 45%"
            "managed a team" → "managed a team of 6 engineers"
- Keep metrics realistic and consistent with the seniority level and industry described.
- Do NOT change anything outside of Responsibilities fields (skills, titles, dates, technologies, education are untouched).
- Preserve the exact semicolon-separated format of the original text.
- Return only the rewritten resume text with no explanation or preamble.

Resume:
{resume}"""

print("✓ Rewrite prompts defined")

### Generate Quantification Variants

In [ ]:
def generate_quant_variant(row, target_col):
    resume_text = row[target_col]
    prompt = STRIP_PROMPT if row['Quantification'] == 'quantified' else ENHANCE_PROMPT
    return call_llm(prompt.format(resume=resume_text))

results_elite    = []
results_nonelite = []

for idx, row in df_resumes.iterrows():
    print(f"Processing resume {idx + 1}/{len(df_resumes)}...", end="\r")
    results_elite.append(generate_quant_variant(row, 'Resume_Elite'))
    time.sleep(1)
    results_nonelite.append(generate_quant_variant(row, 'Resume_NonElite'))
    time.sleep(1)

df_resumes['Resume_Elite_AltQuant']    = results_elite
df_resumes['Resume_NonElite_AltQuant'] = results_nonelite

print(f"\n✓ Quantification variants generated for {len(df_resumes)} resumes")

### Reshape to Final 4-Variant Format

In [ ]:
rows = []

for _, row in df_resumes.iterrows():
    base = {
        'Category'           : row['Category'],
        'Gender'             : row['Gender'],
        'Ethnicity'          : row['Ethnicity'],
        'Base_Quantification': row['Quantification'],
    }
    if row['Quantification'] == 'quantified':
        hq_elite, lq_elite       = row['Resume_Elite'],    row['Resume_Elite_AltQuant']
        hq_nonelite, lq_nonelite = row['Resume_NonElite'], row['Resume_NonElite_AltQuant']
    else:
        lq_elite, hq_elite       = row['Resume_Elite'],    row['Resume_Elite_AltQuant']
        lq_nonelite, hq_nonelite = row['Resume_NonElite'], row['Resume_NonElite_AltQuant']

    rows.append({**base, 'Credentials': 'Elite',     'Quantification': 'quantified',     'Resume': hq_elite})
    rows.append({**base, 'Credentials': 'Elite',     'Quantification': 'non-quantified', 'Resume': lq_elite})
    rows.append({**base, 'Credentials': 'Non-Elite', 'Quantification': 'quantified',     'Resume': hq_nonelite})
    rows.append({**base, 'Credentials': 'Non-Elite', 'Quantification': 'non-quantified', 'Resume': lq_nonelite})

df_final = pd.DataFrame(rows)

print(f"✓ Final dataset: {len(df_final)} rows (30 resumes × 4 variants)")
display(df_final.groupby(['Credentials', 'Quantification']).size().rename('Count').reset_index())

### Quantification Sanity Check

In [ ]:
sample_category = df_final['Category'].iloc[0]  # ← change to spot-check a different resume

sample = df_final[df_final['Category'] == sample_category][
    ['Credentials', 'Quantification', 'Resume']
].copy()
sample['Resume_Preview'] = sample['Resume'].str[:300] + '...'

print(f"=== Quantification Sanity Check: {sample_category} ===")
display(sample[['Credentials', 'Quantification', 'Resume_Preview']])

### Save and Download

In [ ]:
output_filename = 'resumes_final_variants.csv'
df_final.to_csv(output_filename, index=False)

print(f"✓ Saved to {output_filename}")
files.download(output_filename)